# DBLP-Mining (IF4041) — Google Colab Helper: Download, Parse & Preprocess

Notebook mandiri (*self-contained*) untuk mengunduh `dblp.xml.gz` dari server DBLP dengan kecepatan penuh server Google Colab, menjalankan parsing XML (`inproceedings` 2015–2025 + statistik seluruh tipe publikasi XML), serta preprocessing sesuai `src/parser.py` dan `src/preprocessing.py`.

**Hasil akhir** yang perlu diunduh kembali ke laptop hanyalah file hasil proses (atau satu arsip `dblp_processed_bundle.zip` ~200 MB), sehingga tidak perlu mengunduh `dblp.xml.gz` 1.07 GiB lewat jaringan lokal.

### Cara Pakai:
1. Upload file `.ipynb` ini ke https://colab.research.google.com/
2. Klik **Runtime → Run all**.
3. Setelah selesai, unduh `dblp_processed_bundle.zip` dari Google Drive (atau langsung dari sel terakhir) lalu ekstrak ke folder `DBLP-Mining_IF4041/` di laptop Anda.

In [ ]:
# 1. Install dependensi yang diperlukan
!pip -q install lxml pyarrow tqdm pandas

In [ ]:
# 2. (Opsional) Mount Google Drive agar hasil tersimpan aman & bisa diunduh dengan kecepatan penuh
USE_GOOGLE_DRIVE = True
DRIVE_OUT_DIR = "/content/drive/MyDrive/DBLP-Mining_IF4041"

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
    print("Google Drive ready:", DRIVE_OUT_DIR)

In [ ]:
# 3. Unduh dblp.dtd & dblp.xml.gz ke disk lokal Colab (/content/data/raw/) + catat SHA-256
import hashlib
import json
import os
import subprocess
from datetime import datetime, timezone

RAW_DIR = "/content/data/raw"
PROC_DIR = "/content/data/processed"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROC_DIR, exist_ok=True)

def sha256_file(path: str, chunk_size: int = 4 * 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(chunk_size):
            h.update(chunk)
    return h.hexdigest()

MIRRORS = ["https://dblp.org/xml", "https://dblp.uni-trier.de/xml"]
metadata = {"downloaded_at_utc": datetime.now(timezone.utc).isoformat(), "files": {}}

for fname in ("dblp.dtd", "dblp.xml.gz"):
    dest = os.path.join(RAW_DIR, fname)
    downloaded_url = None
    for base in MIRRORS:
        url = f"{base}/{fname}"
        print(f"Downloading {url} -> {dest}")
        res = subprocess.run(["curl", "-L", "-C", "-", "--fail", "-o", dest, url])
        if res.returncode == 0:
            downloaded_url = url
            break
    if downloaded_url is None:
        raise RuntimeError(f"Gagal mengunduh {fname} dari semua mirror")
    size = os.path.getsize(dest)
    metadata["files"][fname] = {
        "filename": fname,
        "url": downloaded_url,
        "size_bytes": size,
        "sha256": sha256_file(dest),
    }
    print(f"OK {fname}: {size / (1024*1024):.2f} MiB | sha256={metadata['files'][fname]['sha256'][:16]}...")

meta_path = os.path.join(RAW_DIR, "metadata.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)
print("Saved metadata to", meta_path)

In [ ]:
# 4. Parsing XML streaming (identik dengan src/parser.py): hitung semua tipe publikasi & ekstrak inproceedings 2015-2025
import gzip
from collections import Counter
import pandas as pd
from lxml import etree
from tqdm.auto import tqdm

PUBLICATION_TAGS = (
    "article", "inproceedings", "proceedings", "book",
    "incollection", "phdthesis", "mastersthesis", "www", "data",
)
YEAR_MIN, YEAR_MAX = 2015, 2025

records = []
pub_type_counts = Counter()

orig_cwd = os.getcwd()
os.chdir(RAW_DIR)
try:
    with gzip.open("dblp.xml.gz", "rb") as f:
        context = etree.iterparse(
            f,
            events=("end",),
            tag=PUBLICATION_TAGS,
            load_dtd=True,
            resolve_entities=True,
            huge_tree=True,
        )
        for _, elem in tqdm(context, desc="Parsing DBLP XML"):
            tag = elem.tag
            pub_type_counts[tag] += 1
            if tag == "inproceedings":
                year_text = elem.findtext("year")
                if year_text and year_text.isdigit():
                    year = int(year_text)
                    if YEAR_MIN <= year <= YEAR_MAX:
                        authors = [a.text.strip() for a in elem.findall("author") if a.text]
                        title_el = elem.find("title")
                        title = "".join(title_el.itertext()).strip() if title_el is not None else None
                        venue = elem.findtext("booktitle")
                        venue = venue.strip() if venue else None
                        key = elem.get("key")
                        records.append({
                            "key": key,
                            "title": title,
                            "authors": authors,
                            "year": year,
                            "venue": venue,
                        })
            elem.clear()
            parent = elem.getparent()
            if parent is not None:
                while elem.getprevious() is not None:
                    del parent[0]
finally:
    os.chdir(orig_cwd)

raw_parquet_path = os.path.join(PROC_DIR, "dblp_2015_2025_raw.parquet")
df_raw = pd.DataFrame(records)
df_raw.to_parquet(raw_parquet_path, index=False)
del records

pub_types_summary = {
    "total_records": int(sum(pub_type_counts.values())),
    "by_type": {k: int(v) for k, v in pub_type_counts.most_common()},
    "extracted_inproceedings_range": [YEAR_MIN, YEAR_MAX],
    "extracted_inproceedings_count": int(len(df_raw)),
}
pub_types_path = os.path.join(PROC_DIR, "xml_publication_types.json")
with open(pub_types_path, "w", encoding="utf-8") as f:
    json.dump(pub_types_summary, f, indent=2)

print(f"Total DBLP XML records scanned : {pub_types_summary['total_records']:,}")
print(f"Extracted inproceedings 2015-25: {len(df_raw):,}")
print("By type:", pub_types_summary["by_type"])

In [ ]:
# 5. Preprocessing & Normalisasi (identik dengan src/preprocessing.py)
import re
import unicodedata

VENUE_ALIASES = {"nips": "NeurIPS"}

def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", text)).strip()

def normalize_author_list(authors) -> list[str]:
    if authors is None:
        return []
    seen, cleaned = set(), []
    for author in authors:
        norm = normalize_text(author)
        if norm and norm not in seen:
            seen.add(norm)
            cleaned.append(norm)
    return cleaned

def normalize_venue_base(venue: str) -> str:
    v = normalize_text(venue)
    v = re.sub(r"\s*@\s*", "@", v)
    v = re.sub(r"\((Workshops?)\)", r"\1", v, flags=re.IGNORECASE)
    v = re.sub(r"\s+", " ", v).strip()
    alias = VENUE_ALIASES.get(v.lower())
    return alias if alias else v

def build_canonical_venue_map(venues: pd.Series) -> dict[str, str]:
    counts = Counter(venues.dropna())
    best_by_lower = {}
    for form, cnt in counts.items():
        key = form.lower()
        prev = best_by_lower.get(key)
        if prev is None or cnt > prev[0] or (cnt == prev[0] and form < prev[1]):
            best_by_lower[key] = (cnt, form)
    return {k: v[1] for k, v in best_by_lower.items()}

def normalize_title_for_dedup(title: str) -> str:
    t = normalize_text(title).lower()
    t = re.sub(r"[^\w\s]", "", t)
    return re.sub(r"\s+", " ", t).strip()

df = df_raw.copy()
stats = {"raw_records": int(len(df))}

df["year"] = pd.to_numeric(df["year"], errors="coerce")
valid_year_mask = df["year"].notna() & df["year"].between(YEAR_MIN, YEAR_MAX)
stats["dropped_invalid_year"] = int((~valid_year_mask).sum())
df = df[valid_year_mask].copy()
df["year"] = df["year"].astype("int32")

df["key"] = df["key"].fillna("").map(normalize_text)
df["title"] = df["title"].fillna("").map(normalize_text)
df["venue"] = df["venue"].fillna("").map(normalize_text)

missing_key_title = (df["key"] == "") | (df["title"] == "")
stats["dropped_missing_key_or_title"] = int(missing_key_title.sum())
df = df[~missing_key_title].copy()

missing_venue = df["venue"] == ""
stats["dropped_missing_venue"] = int(missing_venue.sum())
df = df[~missing_venue].copy()

df["authors"] = df["authors"].map(normalize_author_list)
empty_authors = df["authors"].map(len) == 0
stats["dropped_empty_authors"] = int(empty_authors.sum())
df = df[~empty_authors].copy()

df["venue"] = df["venue"].map(normalize_venue_base)
canonical_map = build_canonical_venue_map(df["venue"])
df["venue"] = df["venue"].str.lower().map(canonical_map)

before_key_dedup = len(df)
df = df.drop_duplicates(subset=["key"], keep="first").copy()
stats["dropped_duplicate_key"] = int(before_key_dedup - len(df))

df["_title_norm"] = df["title"].map(normalize_title_for_dedup)
before_content_dedup = len(df)
df = df.drop_duplicates(subset=["_title_norm", "year", "venue"], keep="first").copy()
stats["dropped_duplicate_title_year_venue"] = int(before_content_dedup - len(df))
df = df.drop(columns=["_title_norm"]).reset_index(drop=True)

stats["cleaned_records"] = int(len(df))
stats["total_removed"] = int(stats["raw_records"] - stats["cleaned_records"])
stats["unique_authors"] = int(df["authors"].explode().nunique())
stats["unique_venues"] = int(df["venue"].nunique())
stats["year_distribution"] = {int(k): int(v) for k, v in df["year"].value_counts().sort_index().items()}
stats["top_20_venues"] = {str(k): int(v) for k, v in df["venue"].value_counts().head(20).items()}

clean_parquet_path = os.path.join(PROC_DIR, "dblp_2015_2025_clean.parquet")
summary_json_path = os.path.join(PROC_DIR, "preprocessing_summary.json")
df.to_parquet(clean_parquet_path, index=False)
with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)

print(f"Cleaned records: {stats['cleaned_records']:,}")
print(f"Unique authors : {stats['unique_authors']:,}")
print(f"Unique venues  : {stats['unique_venues']:,}")

In [ ]:
# 6. Paketkan seluruh hasil proses ke satu ZIP (~200 MB) & salin ke Google Drive
import shutil
import zipfile

bundle_path = "/content/dblp_processed_bundle.zip"
files_to_bundle = [
    ("data/raw/metadata.json", os.path.join(RAW_DIR, "metadata.json")),
    ("data/processed/xml_publication_types.json", os.path.join(PROC_DIR, "xml_publication_types.json")),
    ("data/processed/preprocessing_summary.json", os.path.join(PROC_DIR, "preprocessing_summary.json")),
    ("data/processed/dblp_2015_2025_clean.parquet", os.path.join(PROC_DIR, "dblp_2015_2025_clean.parquet")),
]
# Catatan: dblp_2015_2025_raw.parquet tidak wajib karena hanya dipakai oleh preprocessing.py;
# hapus tanda komentar di bawah jika ingin menyertakannya juga (+~200 MB):
# files_to_bundle.append(("data/processed/dblp_2015_2025_raw.parquet", os.path.join(PROC_DIR, "dblp_2015_2025_raw.parquet")))

with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_STORED) as zf:
    for arcname, src in files_to_bundle:
        zf.write(src, arcname=arcname)

print(f"Bundle created: {bundle_path} ({os.path.getsize(bundle_path)/(1024*1024):.1f} MiB)")

if USE_GOOGLE_DRIVE:
    drive_bundle = os.path.join(DRIVE_OUT_DIR, "dblp_processed_bundle.zip")
    shutil.copy2(bundle_path, drive_bundle)
    print("Copied bundle to Google Drive:", drive_bundle)
    print("Silakan unduh file tersebut dari Google Drive (full speed), lalu ekstrak ke folder root DBLP-Mining_IF4041/")

In [ ]:
# 7. (Opsional) Unduh langsung lewat browser tanpa Google Drive
# Hapus komentar di bawah jika ingin download langsung dari Colab:
# from google.colab import files
# files.download("/content/dblp_processed_bundle.zip")